Project Structure

In [ ]:
# InterestRateImpactAnalyzer/
# │
# ├── data/
# │   ├── raw/                     # Raw downloaded datasets (World Bank, IMF, etc.)
# │   ├── processed/               # Cleaned and transformed datasets
# │   └── external/                # Any additional API or third-party data
# │
# ├── notebooks/
# │   ├── EDA.ipynb                # Exploratory Data Analysis
# │   ├── preprocessing.ipynb      # Data cleaning, transformation
# │   ├── model_linear_reg.ipynb   # Linear Regression model notebook
# │   ├── model_xgboost.ipynb      # XGBoost model notebook
# │   └── model_lstm.ipynb         # LSTM model notebook
# │
# ├── src/
# │   ├── __init__.py
# │   ├── data_loader.py           # Functions for loading and preprocessing data
# │   ├── feature_engineering.py   # Feature creation/transformation scripts
# │   ├── model_trainer.py         # Model training scripts (modular)
# │   ├── predictor.py             # Functions for predictions
# │   └── visualizer.py            # Plotting and graph utilities
# │
# ├── dashboard/ (optional)
# │   ├── app.py                   # Dashboard app (e.g., using Streamlit/Dash)
# │   └── components/              # UI and charts
# │
# ├── reports/
# │   ├── figures/                 # Graphs, charts for report
# │   └── final_report.pdf         # Final project documentation
# │
# ├── models/
# │   ├── linear_reg_model.pkl
# │   ├── xgboost_model.pkl
# │   └── lstm_model.h5
# │
# ├── requirements.txt             # All Python dependencies
# ├── README.md                    # Project overview and setup instructions
# ├── budget.docx                  # Project budget
# └── main.py                      # Entry point script for model running or CLI


Notebooks

In [ ]:
# Eda.py

# %% [markdown]
# # Exploratory Data Analysis (EDA)
# 
# This notebook explores trends in interest rates, GDP, inflation, and other economic indicators.

# %%
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_raw_data
from src.visualizer import plot_series

# %% [markdown]
# ## 1. Load Raw Data

# %%
# Load datasets
interest_df = load_raw_data("interest_rates")
gdp_df = load_raw_data("gdp")
inflation_df = load_raw_data("inflation")

# Merge data
merged_df = pd.merge(interest_df, gdp_df, on='date')
merged_df = pd.merge(merged_df, inflation_df, on='date')
merged_df['date'] = pd.to_datetime(merged_df['date'])

# %% [markdown]
# ## 2. Basic Statistics

# %%
print("Summary Statistics:")
display(merged_df.describe())

print("\nMissing Values:")
display(merged_df.isnull().sum())

# %% [markdown]
# ## 3. Time Series Trends

# %%
# Plot interest rates vs GDP
plt.figure(figsize=(12, 6))
sns.lineplot(x='date', y='interest_rate', data=merged_df, label='Interest Rate')
sns.lineplot(x='date', y='gdp', data=merged_df, label='GDP', alpha=0.5)
plt.title("Interest Rates vs GDP Over Time")
plt.savefig("../reports/figures/interest_vs_gdp.png")
plt.show()

# %% [markdown]
# ## 4. Correlation Analysis

# %%
corr_matrix = merged_df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix of Economic Indicators")
plt.savefig("../reports/figures/correlation_heatmap.png")
plt.show()

# %% [markdown]
# ## 5. Distribution Plots

# %%
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.histplot(merged_df['interest_rate'], kde=True, ax=axes[0,0])
sns.histplot(merged_df['gdp'], kde=True, ax=axes[0,1])
sns.boxplot(x='inflation_rate', data=merged_df, ax=axes[1,0])
plt.tight_layout()
plt.savefig("../reports/figures/distributions.png")
plt.show()

# %% [markdown]
# ## 6. Lag Analysis

# %%
# Create lagged features for analysis
merged_df['interest_rate_lag1'] = merged_df['interest_rate'].shift(1)
merged_df['gdp_change'] = merged_df['gdp'].pct_change() * 100

# Scatter plot of lagged effect
plt.figure(figsize=(10, 6))
sns.scatterplot(x='interest_rate_lag1', y='gdp_change', data=merged_df)
plt.title("Lagged Interest Rate vs GDP Growth")
plt.savefig("../reports/figures/lag_effect.png")
plt.show()

In [ ]:
# 2.preprocessing.py

# %% [markdown]
# # Data Preprocessing
# 
# This notebook cleans raw data and creates features for modeling.

# %%
import pandas as pd
from src.data_loader import load_raw_data, save_processed_data
from src.feature_engineering import create_features

# %% [markdown]
# ## 1. Load Raw Data
# %%
interest_df = load_raw_data("interest_rates")
gdp_df = load_raw_data("gdp")
inflation_df = load_raw_data("inflation")

# Merge datasets
merged_df = pd.merge(interest_df, gdp_df, on='date')
merged_df = pd.merge(merged_df, inflation_df, on='date')

# %% [markdown]
# ## 2. Handle Missing Values
# %%
merged_df.dropna(inplace=True)

# %% [markdown]
# ## 3. Feature Engineering
# %%
processed_df = create_features(merged_df)
save_processed_data(processed_df, "processed_data.csv")

# %% [markdown]
# ## 4. Save Processed Data
# %%
print("Processed Data Shape:", processed_df.shape)
processed_df.head()

In [ ]:
# 3.linear_regression.py

# This code is used for predicting GDP using a linear regression model based on interest rates and inflation data.

# %% [markdown]
# # Linear Regression Model
# 
# Trains and evaluates a baseline linear regression model.

# %%
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.data_loader import load_processed_data
from src.model_trainer import train_linear_model
from src.visualizer import plot_series

# %% [markdown]
# ## 1. Load Processed Data
# %%
df = load_processed_data("processed_data.csv")
X = df[['interest_rate_lag1', 'inflation_rolling3']]
y = df['gdp']

# %% [markdown]
# ## 2. Train Model
# %%
model = train_linear_model(X, y)
print("Model coefficients:", model.coef_)

# %% [markdown]
# ## 3. Evaluate Predictions
# %%
y_pred = model.predict(X)
plt.figure(figsize=(10, 6))
plt.scatter(y, y_pred, alpha=0.5)
plt.xlabel("Actual GDP")
plt.ylabel("Predicted GDP")
plt.title("Linear Regression: Actual vs Predicted")
plt.savefig("../reports/figures/linear_regression_performance.png")


# Below code is for other predictions

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Load the dataset
df = pd.read_csv("processed_data.csv")  # your cleaned dataset

# Features (input)
X = df[['interest_rate']]  # only interest rate as input

# Targets (outputs to be predicted)
Y = df[['gdp', 'economic_growth', 'inflation', 'savings', 'loan_rates']]

# Train the model
model = LinearRegression()
model.fit(X, Y)

# Print model coefficients
print("Intercept:", model.intercept_)
print("Coefficients (per output):")
for i, col in enumerate(Y.columns):
    print(f"{col}: {model.coef_[0][i]}")

# Predict values
Y_pred = model.predict(X)

# Plot actual vs predicted for each economic factor
for i, col in enumerate(Y.columns):
    plt.figure(figsize=(8, 5))
    plt.scatter(Y[col], Y_pred[:, i], alpha=0.5)
    plt.xlabel(f"Actual {col}")
    plt.ylabel(f"Predicted {col}")
    plt.title(f"Interest Rate vs {col}")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"../reports/figures/interest_rate_vs_{col}.png")
    plt.show()


ModuleNotFoundError: No module named 'matplotlib'